# 从模型调用到 Agent Runtime：全文互动练习

这是一份跟随《深入 OpenAI Agents SDK：从一次模型调用到 Agent Runtime》推进的实验讲义。
我们沿一条天气请求阅读 SDK，再逐步加入工具、交接、状态、生命周期与边界实验。

**你要做的事：先预测 → 运行 → 改一个参数 → 对照源码解释。**

- 全文必做单元均使用 SDK 的 `ScriptedModel`，无需 API Key，不调用真实大模型。
- 模型输出、天气与退款数据都是预设的。我们验证的是 SDK 控制流，而不是模型推理质量或真实天气。
- 第 7 章的 Trace 只写入内存，不发送到 OpenAI 平台；最后的真实模型尝试默认关闭。
- 发行包固定为 `openai-agents==0.24.0`；源码链接固定 commit `17d994bd3045dae71acda676a4ef4762ed8ecd94`。
  固定发行包版本不等于已经证明 wheel 与研究提交完全一致；运行时会显示实际加载路径。

### 如何一起跟练

1. 在 Colab 打开后，先保存一份副本，方便修改自己的参数。
2. 从上往下运行。安装完成后先做环境检查，再进入章节。
3. 修改题目的参数后，重新运行该节**完整代码单元**，不要只重复最后一行：模拟模型的步骤会被消耗。
4. `assert` 是检查预期的断言；失败时请对照实际输出。若故意改变条件，也要相应调整预期。
5. Python 中 `await` 用于等待异步结果；Notebook 可直接写 `await`。不要在 Notebook 中嵌套 `Runner.run_sync()`。

**现场主线：** 0 环境 → 1 角色 → 2 循环 → 3 模型边界 → 4 工具 → 5 交接 → 6 审批 → 7 hooks → 8 边界 → 9–10 回顾。
第 4 章的工具错误扩展、第 6 章 Context/Session、第 7 章 Streaming/Trace 可按现场节奏留作课后实操。

[1 角色](#scrollTo=chapter-01) · [2 循环](#scrollTo=chapter-02) · [3 模型](#scrollTo=chapter-03) · [4 工具](#scrollTo=chapter-04) · [5 Handoff](#scrollTo=chapter-05) · [6 状态](#scrollTo=chapter-06) · [7 生命周期](#scrollTo=chapter-07) · [8 边界](#scrollTo=chapter-08) · [9 设计](#scrollTo=chapter-09) · [10 总结](#scrollTo=chapter-10)

## 0 · 环境准备

Colab 不需要你在电脑上安装 Python。运行下面的**代码单元格**，将 SDK 安装到当前运行时。
安装输出若报错，请先解决安装问题；不要直接跳到 `import agents`。
如果安装后仍找不到模块，重启运行时后再运行版本检查。

安装行前的 `%pip` 是 Notebook 命令；它应放在代码单元，不是 Markdown 说明块中。

In [1]:
%pip install -q "openai-agents==0.24.0"


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: /tmp/agents-sdk-full-workshop-20261112/env/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path
from importlib.metadata import version
from importlib.util import find_spec

print("Python:", sys.version.split()[0])
print("Notebook 内核:", sys.executable)
# 本地源码曾移动过时，可以从练习本旁边的源码目录加载；Colab 通常不走此分支。
if find_spec("agents") is None:
    source = Path.cwd() / "openai-agents-python" / "src"
    if (source / "agents" / "__init__.py").is_file():
        sys.path.insert(0, str(source))
        print("本地源码模式:", source)

import agents
print("openai-agents:", version("openai-agents"))
print("SDK 加载路径:", agents.__file__)
assert version("openai-agents") == "0.24.0", "请先运行安装单元"
from agents.testing import ModelStep, ScriptedModel
print("环境检查通过：后面的主线不需要 API Key。")

Python: 3.11.11
Notebook 内核: /tmp/agents-sdk-full-workshop-20261112/env/bin/python


openai-agents: 0.24.0
SDK 加载路径: /tmp/agents-sdk-full-workshop-20261112/env/lib/python3.11/site-packages/agents/__init__.py
环境检查通过：后面的主线不需要 API Key。


### 一次准备：模拟响应与天气工具

下面封装少量辅助代码，后面每个实验都重新创建 Agent 和模拟模型。
`ModelStep` 描述一次预设响应；`ScriptedModel` 会按顺序消费它们，并在 `calls` 中记录实际模型调用。

你只需先认识 `function_call`（请求执行工具）与普通 `message`（模型消息）。
辅助函数**没有重新实现 Runner**；后面的执行循环与工具处理仍由真实 SDK 完成。

**看代码时，只需先认识这几个写法：**

| Python 写法 | 在这里表示什么 |
|---|---|
| `def` / `async def` | 定义普通函数 / 异步函数 |
| `await` | 等待异步调用完成，类似 JavaScript 的 await |
| `class X(Base)` | 定义继承 Base 的类，类似 JavaScript 的 extends |
| `@function_tool` | 用装饰器把函数包装成工具对象 |
| `city: str` | 参数 city 的类型注解；工具包装器会据此建立校验模型 |
| `assert 条件` | 检查实验预期；失败会显示断言错误 |


In [3]:
import inspect
import itertools
import json
from dataclasses import dataclass
from agents import Agent, FunctionTool, Handoff, MaxTurnsExceeded, RunConfig, RunHooks, Runner, function_tool
from openai.types.responses import ResponseFunctionToolCall, ResponseOutputMessage
from openai.types.responses.response_output_text import ResponseOutputText

_ids = itertools.count(1)
OFFLINE = RunConfig(tracing_disabled=True)

def tool_call(name, arguments, call_id=None):
    call_id = call_id or f"call_{next(_ids)}"
    return ResponseFunctionToolCall(
        id=call_id, call_id=call_id, type="function_call", name=name,
        arguments=json.dumps(arguments, ensure_ascii=False),
    )

def final_message(text):
    return ResponseOutputMessage(
        id=f"msg_{next(_ids)}", type="message", role="assistant", status="completed",
        content=[ResponseOutputText(text=text, type="output_text", annotations=[], logprobs=[])],
    )

def make_weather_agent(steps, **options):
    events = []
    @function_tool
    def get_weather(city: str) -> str:
        """Return fixed illustrative weather for a city."""
        events.append(city)
        return f"{city}: sunny, 14-20C"
    model = ScriptedModel(steps)
    agent = Agent(name="Weather Agent", instructions="查询天气后回答。", tools=[get_weather], model=model, **options)
    return agent, model, events

def tool_outputs(result):
    return [item.output for item in result.new_items if item.type == "tool_call_output_item"]

def show_source(obj, keyword=None, lines=12):
    source_lines, first = inspect.getsourcelines(obj)
    offset = next((i for i, line in enumerate(source_lines) if keyword in line), 0) if keyword else 0
    print(f"{inspect.getfile(obj)}:{first + offset}")
    print("".join(f"{first+i:4d}  {line}" for i, line in enumerate(source_lines[offset:offset+lines], offset)))

print("辅助代码已就绪。")

辅助代码已就绪。


## 第 1 章 · 为什么需要 SDK？Agent 和 Runner 各自做什么？

**对应笔记：第1章。** 从配置对象开始。

**先预测：** 创建 Agent 后，天气函数已经执行了吗？模型已经被调用了吗？

**源码入口（固定提交）：**
- [Agent 定义](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/agent.py)
- [Runner 公开入口](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/run.py)

In [4]:
config_agent, config_model, config_events = make_weather_agent([
    ModelStep(output=[final_message("这一步暂时不会执行")])
])
print("Agent:", config_agent.name)
print("注册的工具:", [tool.name for tool in config_agent.tools])
print("工具对象类型:", type(config_agent.tools[0]).__name__)
print("模型调用次数:", len(config_model.calls))
print("工具执行记录:", config_events)
assert isinstance(config_agent.tools[0], FunctionTool)
assert len(config_model.calls) == 0 and config_events == []

Agent: Weather Agent
注册的工具: ['get_weather']
工具对象类型: FunctionTool
模型调用次数: 0
工具执行记录: []


**观察：** Agent 声明模型、指令和工具；这时还没有执行任务。`Runner.run()` 才进入运行过程。

**动手改：** 修改 Agent 名称或 Instructions，重新运行上一个单元，确认只是配置变化。
**解释给同桌：** 模型、Agent 配置和 SDK 执行器分别负责什么？

## 第 2 章 · Runner 如何推进 Agent Loop？

**对应笔记：第2章。** 从真实 Runner 执行一条天气查询。

**先预测：** 一次工具调用后需要几次模型调用？第二次输入里会有什么？

**源码入口（固定提交）：**
- [循环入口](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/run.py)
- [单轮执行](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/run_internal/run_loop.py)
- [NextStep](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/run_internal/run_steps.py)

In [5]:
CITY = "Shanghai"  # 动手改成 Beijing，再运行整个单元
loop_agent, loop_model, loop_events = make_weather_agent([
    ModelStep(output=[tool_call("get_weather", {"city": CITY})]),
    ModelStep(output=[final_message("这是预设的最终建议，不代表真实推理。")]),
])
loop_result = await Runner.run(loop_agent, f"{CITY} 天气怎么样？", run_config=OFFLINE)
print("模型调用次数:", len(loop_model.calls))
print("工具执行记录:", loop_events)
print("运行项类型:", [item.type for item in loop_result.new_items])
print("第二次模型输入:", json.dumps(loop_model.calls[1].input, ensure_ascii=False, indent=2))
print("最终输出:", loop_result.final_output)
assert len(loop_model.calls) == 2 and loop_events == [CITY]
assert any(item.get("type") == "function_call_output" for item in loop_model.calls[1].input)

模型调用次数: 2
工具执行记录: ['Shanghai']
运行项类型: ['tool_call_item', 'tool_call_output_item', 'message_output_item']
第二次模型输入: [
  {
    "content": "Shanghai 天气怎么样？",
    "role": "user"
  },
  {
    "arguments": "{\"city\": \"Shanghai\"}",
    "call_id": "call_2",
    "name": "get_weather",
    "type": "function_call",
    "id": "call_2"
  },
  {
    "call_id": "call_2",
    "output": "Shanghai: sunny, 14-20C",
    "type": "function_call_output"
  }
]
最终输出: 这是预设的最终建议，不代表真实推理。


**观察：** SDK 在两次模型调用之间执行工具，把请求与结果加入历史。`call_id` 关联一次具体调用；它不是工具名称。

现在对照源码看这条路径：`Runner.run → AgentRunner.run → _run_impl → run_single_turn`。
`AgentRunner` 是内部实验性实现，下面只读取它的源码，不直接调用或继承它。

In [6]:
from agents.run import AgentRunner
show_source(Runner.run, "runner = DEFAULT_AGENT_RUNNER", lines=10)
show_source(AgentRunner.run, "return await self._run_impl", lines=5)
show_source(AgentRunner._run_impl, "while True:", lines=8)
show_source(AgentRunner._run_impl, "current_turn += 1", lines=7)

/tmp/agents-sdk-full-workshop-20261112/env/lib/python3.11/site-packages/agents/run.py:349
 349          runner = DEFAULT_AGENT_RUNNER
 350          redacted_error: BaseException | None = None
 351          try:
 352              return await runner.run(
 353                  starting_agent,
 354                  input,
 355                  context=context,
 356                  max_turns=max_turns,
 357                  hooks=hooks,
 358                  run_config=run_config,

/tmp/agents-sdk-full-workshop-20261112/env/lib/python3.11/site-packages/agents/run.py:610
 610                          return await self._run_impl(starting_agent, input, **kwargs)
 611                  except BaseException as error:
 612                      if not _is_error_data_redacted(error):
 613                          raise
 614                      _detach_data_redacted_error_traceback(error)

/tmp/agents-sdk-full-workshop-20261112/env/lib/python3.11/site-packages/agents/run.py:1032
1032              

## 第 3 章 · 模型如何被找到、调用与解析？

**对应笔记：第3章。** 用自定义 Provider 观察 Model 边界，仍不访问真实 API。

**先预测：** Agent.model 与 RunConfig.model 同时指定，哪个名称交给 Provider？

**源码入口（固定提交）：**
- [配置优先级](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/run_internal/turn_preparation.py)
- [Provider / Model 接口](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/models/interface.py)
- [响应解析](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/run_internal/turn_resolution.py)

In [7]:
from agents.models.interface import ModelProvider

class RecordingProvider(ModelProvider):
    def __init__(self, model):
        self.model = model
        self.requested_names = []
    def get_model(self, model_name):
        self.requested_names.append(model_name)
        return self.model

provider_model = ScriptedModel([ModelStep(output=[final_message("预设模型响应")])])
provider = RecordingProvider(provider_model)
provider_agent = Agent(name="Provider Demo", model="study/from-agent")
provider_result = await Runner.run(
    provider_agent, "你好",
    run_config=RunConfig(model="study/from-run-config", model_provider=provider, tracing_disabled=True),
)
print("Provider 收到的名称:", provider.requested_names)
print("实际调用的模型对象:", type(provider_model).__name__)
print("模型边界收到的输入:", provider_model.calls[0].input)
assert provider.requested_names == ["study/from-run-config"]

Provider 收到的名称: ['study/from-run-config']
实际调用的模型对象: ScriptedModel
模型边界收到的输入: [{'content': '你好', 'role': 'user'}]


**动手改：** 删掉上个单元中的 `model="study/from-run-config"` 配置项后重跑，预测 Provider 将收到什么名称，并同步修改断言。

真实默认 HTTP 路径是 `get_model → OpenAIResponsesModel.get_response → client.responses.create`。
本节替换了 Model，因此没有发送 Responses API 请求。`ScriptedModel` 返回的是统一的响应项目；不能用它证明真实 API 服务或网络正常。

`process_model_response()` 识别操作并匹配工具；真正的 Python 函数执行在下一阶段。

## 第 4 章 · Tool Execution：函数如何变成工具，再真正执行？

**对应笔记：第4章。** 跟随新版第四章：包装 → Schema → 名称匹配 → 参数 → 函数 → 工具结果。

**先预测：** 装饰器是否执行原始函数？模型能看到本地执行回调吗？

**源码入口（固定提交）：**
- [工具包装与执行回调](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/tool.py)
- [工具 Schema](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/function_schema.py)
- [批量工具执行](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/run_internal/tool_execution.py)

In [8]:
registration_events = []
def original_weather(city: str) -> str:
    """查询城市的固定示例天气。"""
    registration_events.append(city)
    return f"{city}：晴天"

wrapped_weather = function_tool(original_weather)
print("原始函数:", type(original_weather).__name__)
print("包装后的对象:", type(wrapped_weather).__name__)
print("名称:", wrapped_weather.name)
print("描述:", wrapped_weather.description)
print("参数 Schema:", json.dumps(wrapped_weather.params_json_schema, ensure_ascii=False, indent=2))
print("SDK 保存了执行入口:", callable(wrapped_weather.on_invoke_tool))
print("装饰时是否已执行:", registration_events)
assert isinstance(wrapped_weather, FunctionTool) and registration_events == []

原始函数: function
包装后的对象: FunctionTool
名称: original_weather
描述: 查询城市的固定示例天气。
参数 Schema: {
  "properties": {
    "city": {
      "title": "City",
      "type": "string"
    }
  },
  "required": [
    "city"
  ],
  "title": "original_weather_args",
  "type": "object",
  "additionalProperties": false
}
SDK 保存了执行入口: True
装饰时是否已执行: []


**观察：** `FunctionTool = 工具描述 + 执行入口`。模型请求只接收名称、描述、参数等声明；本地回调留给 SDK。

**动手改：** 给原函数增加一个有默认值的参数，重新包装，观察 Schema，而不是手工猜测必填字段。

下面改变参数类型。固定 `city: str`，请预测数字参数能否进入函数体。

In [9]:
ARGUMENTS = {"city": 123}  # 课后可改成 {} 或 {"city": "Shanghai"}
bad_agent, bad_model, bad_events = make_weather_agent([
    ModelStep(output=[tool_call("get_weather", ARGUMENTS)]),
    ModelStep(output=[final_message("预设的错误后响应")]),
])
bad_result = await Runner.run(bad_agent, "查询天气", run_config=OFFLINE)
print("工具函数执行记录:", bad_events)
print("SDK 产生的工具输出:", tool_outputs(bad_result))
print("模型调用次数:", len(bad_model.calls))
assert bad_events == []
assert len(bad_model.calls) == 2 and len(tool_outputs(bad_result)) == 1

工具函数执行记录: []
SDK 产生的工具输出: ['An error occurred while running the tool. Please try again.']
模型调用次数: 2


**观察：** 参数处理失败时，函数体没有执行；本例默认错误转换仍生成了模型可见的结果。
如果把参数改成有效字符串，记得把 `bad_events == []` 的预期也改掉。

**再预测：** 参数正确，但工具抛出 `ValueError`，这条默认配置路径会立即终止 Run 吗？

In [10]:
@function_tool
def unstable_tool(value: str) -> str:
    """模拟失败的本地操作。"""
    raise ValueError("模拟数据库故障")

error_model = ScriptedModel([
    ModelStep(output=[tool_call("unstable_tool", {"value": "demo"})]),
    ModelStep(output=[final_message("预设的异常后回答")]),
])
error_agent = Agent(name="Error Demo", tools=[unstable_tool], model=error_model)
error_result = await Runner.run(error_agent, "执行操作", run_config=OFFLINE)
print("工具结果:", tool_outputs(error_result))
print("模型调用次数:", len(error_model.calls))
assert len(error_model.calls) == 2 and len(tool_outputs(error_result)) == 1

工具结果: ['An error occurred while running the tool. Please try again.']
模型调用次数: 2


这个结论只适用于本例的默认错误处理。`failure_error_function=None`、上层错误处理器、超时和取消会改变传播路径。

**接着预测：** `tool_use_behavior="stop_on_first_tool"` 时，还会消耗第二个模型响应吗？

In [11]:
stop_agent, stop_model, stop_events = make_weather_agent([
    ModelStep(output=[tool_call("get_weather", {"city": "Shanghai"})]),
], tool_use_behavior="stop_on_first_tool")
stop_result = await Runner.run(stop_agent, "查天气", run_config=OFFLINE)
print("模型调用次数:", len(stop_model.calls))
print("最终输出:", stop_result.final_output)
assert len(stop_model.calls) == 1 and stop_events == ["Shanghai"]
assert stop_result.final_output == "Shanghai: sunny, 14-20C"

模型调用次数: 1
最终输出: Shanghai: sunny, 14-20C


**本章串起来：** Python 函数 → FunctionTool → 模型工具声明 → Function Call → ToolRunFunction → 执行计划/批量执行器 → 回调 → Python 函数 → ToolCallOutputItem → NextStep。

工具结果可以交回模型，也可以按配置直接作为最终输出。不要把“工具结束”与“Run 结束”当作同一件事。

## 第 5 章 · Handoff：怎样交接控制权？

**对应笔记：第5章。** 先运行交接，再用 as_tool 对照。

**先预测：** triage 交给 specialist 后，最后的 Agent 是谁？

**源码入口（固定提交）：**
- [Handoff 定义与工厂](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/handoffs/__init__.py)
- [交接识别与执行](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/run_internal/turn_resolution.py)
- [Agent.as_tool](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/agent.py)

In [12]:
class RecordHandoff(RunHooks):
    def __init__(self): self.events = []
    async def on_handoff(self, context, from_agent, to_agent):
        self.events.append(f"{from_agent.name} -> {to_agent.name}")

specialist_model = ScriptedModel([ModelStep(output=[final_message("预设的专家回答")])])
specialist = Agent(name="specialist", model=specialist_model)
transfer_name = Handoff.default_tool_name(specialist)
triage_model = ScriptedModel([ModelStep(output=[tool_call(transfer_name, {})])])
triage = Agent(name="triage", model=triage_model, handoffs=[specialist])
handoff_hooks = RecordHandoff()
handoff_result = await Runner.run(triage, "请交接给专家", hooks=handoff_hooks, run_config=OFFLINE)
print("交接请求名:", transfer_name)
print("交接事件:", handoff_hooks.events)
print("最终 Agent:", handoff_result.last_agent.name)
print("两个 Agent 各自的模型调用数:", len(triage_model.calls), len(specialist_model.calls))
assert handoff_hooks.events == ["triage -> specialist"]
assert handoff_result.last_agent.name == "specialist"

交接请求名: transfer_to_specialist
交接事件: ['triage -> specialist']
最终 Agent: specialist
两个 Agent 各自的模型调用数: 1 1


Handoff 是独立对象，SDK 将它转换成模型能理解的 Function Tool 声明。它没有变成 SDK 的 FunctionTool 子类。

**对照预测：** 如果专家通过 `as_tool()` 被调用，最终 Agent 还会变成专家吗？

In [13]:
expert_model = ScriptedModel([ModelStep(output=[final_message("预设的专家子任务结果")])])
expert = Agent(name="expert", model=expert_model)
expert_tool = expert.as_tool(
    tool_name="ask_expert", tool_description="请求专家完成子任务", run_config=OFFLINE,
)
print("子任务工具 Schema:", expert_tool.params_json_schema)
manager_model = ScriptedModel([
    ModelStep(output=[tool_call("ask_expert", {"input": "分析天气"})]),
    ModelStep(output=[final_message("预设的经理最终回答")]),
])
manager = Agent(name="manager", model=manager_model, tools=[expert_tool])
manager_result = await Runner.run(manager, "咨询专家后回答", run_config=OFFLINE)
print("最终 Agent:", manager_result.last_agent.name)
print("主 Agent / 专家模型调用数:", len(manager_model.calls), len(expert_model.calls))
print("专家工具结果:", tool_outputs(manager_result))
assert manager_result.last_agent.name == "manager"
assert len(manager_model.calls) == 2 and len(expert_model.calls) == 1

子任务工具 Schema: {'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}
最终 Agent: manager
主 Agent / 专家模型调用数: 2 1
专家工具结果: ['预设的专家子任务结果']


**观察：** Handoff 转移后续执行控制权；`as_tool` 可将子 Agent 的结果交回主 Agent 继续。

**动手改：** 更换专家名称后，不要手写猜测 transfer 名称，继续使用 `Handoff.default_tool_name()` 生成它。

## 第 6 章 · Context、Session、RunState 与审批恢复

**对应笔记：第6章。** 区分业务上下文、对话历史和执行现场。

**先预测：** 审批工具暂停时，函数执行次数应为 0 还是 1？恢复会重新调用第一轮模型吗？

**源码入口（固定提交）：**
- [Context](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/run_context.py)
- [Session](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/memory/session.py)
- [RunState](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/run_state.py)

In [14]:
refund_events = []
@function_tool(needs_approval=True)
def refund_order(order_id: str, amount: int) -> str:
    """模拟退款；不连接真实订单或支付系统。"""
    refund_events.append((order_id, amount))
    return f"订单 {order_id} 已模拟退款 {amount} 元"

approval_model = ScriptedModel([
    ModelStep(output=[tool_call("refund_order", {"order_id": "A123", "amount": 500})]),
    ModelStep(output=[final_message("预设的审批后回答")]),
])
approval_agent = Agent(name="Refund Agent", model=approval_model, tools=[refund_order])
paused = await Runner.run(approval_agent, "给订单 A123 退款 500 元", run_config=OFFLINE)
print("待审批数量:", len(paused.interruptions))
print("审批前工具执行记录:", refund_events)
print("审批前模型调用次数:", len(approval_model.calls))
assert len(paused.interruptions) == 1 and refund_events == []

state = paused.to_state()
print("保存的现场类型:", type(state).__name__)
for item in paused.interruptions:
    state.approve(item)  # 本实验显式模拟“人工批准”，真实业务需独立鉴权与审批
resumed = await Runner.run(approval_agent, state, run_config=OFFLINE)
print("恢复后工具执行记录:", refund_events)
print("恢复后模型调用总数:", len(approval_model.calls))
print("最终输出:", resumed.final_output)
assert refund_events == [("A123", 500)] and len(approval_model.calls) == 2
assert not resumed.interruptions

待审批数量: 1
审批前工具执行记录: []
审批前模型调用次数: 1
保存的现场类型: RunState
恢复后工具执行记录: [('A123', 500)]
恢复后模型调用总数: 2
最终输出: 预设的审批后回答


这里证明的是**内存中同一次 Run 的批准与恢复**，不覆盖拒绝、再次审批、跨进程序列化或所有故障。
真实退款仍需要业务幂等和权限边界。脚本固定为恢复后返回消息，不能由此推断真实模型不会再请求退款。

**课后扩展 A：Context。** 下面的业务上下文会自动出现在模型输入里吗？

In [15]:
from agents import RunContextWrapper

@dataclass
class AppContext:
    user_id: str

@function_tool
def get_user_id(ctx: RunContextWrapper[AppContext]) -> str:
    """读取应用提供的用户标识。"""
    return ctx.context.user_id

context_model = ScriptedModel([
    ModelStep(output=[tool_call("get_user_id", {})]),
    ModelStep(output=[final_message("预设的用户查询回答")]),
])
context_agent = Agent[AppContext](name="Context Demo", model=context_model, tools=[get_user_id])
context_result = await Runner.run(context_agent, "查询我的标识", context=AppContext("study_user_001"), run_config=OFFLINE)
first_input = json.dumps(context_model.calls[0].input, ensure_ascii=False)
second_input = json.dumps(context_model.calls[1].input, ensure_ascii=False)
print("第一次输入含用户标识:", "study_user_001" in first_input)
print("工具结果回传后的输入含用户标识:", "study_user_001" in second_input)
assert "study_user_001" not in first_input and "study_user_001" in second_input

第一次输入含用户标识: False
工具结果回传后的输入含用户标识: True


**课后扩展 B：Session。** 在两次独立的 `Runner.run()` 之间保存历史，与中断现场有什么不同？
下面使用内存 SQLite，不创建持久数据库文件。

In [16]:
from agents import SQLiteSession
session = SQLiteSession("study_conversation")
session_model = ScriptedModel([
    ModelStep(output=[final_message("已收到订单 A123（预设）")]),
    ModelStep(output=[final_message("已结合历史回应（预设，不代表真实理解）")]),
])
session_agent = Agent(name="Session Demo", model=session_model)
try:
    await Runner.run(session_agent, "我要查订单 A123", session=session, run_config=OFFLINE)
    await Runner.run(session_agent, "它发货了吗？", session=session, run_config=OFFLINE)
    history_input = json.dumps(session_model.calls[1].input, ensure_ascii=False)
    print("第二次 Run 的模型输入:", history_input)
    print("Session 项目数:", len(await session.get_items()))
    assert "A123" in history_input and "它发货了吗" in history_input
finally:
    session.close()

第二次 Run 的模型输入: [{"content": "我要查订单 A123", "role": "user"}, {"id": "msg_19", "content": [{"annotations": [], "text": "已收到订单 A123（预设）", "type": "output_text", "logprobs": []}], "role": "assistant", "status": "completed", "type": "message"}, {"content": "它发货了吗？", "role": "user"}]
Session 项目数: 4


## 第 7 章 · Guardrails、Streaming、Tracing 与扩展点

**对应笔记：第7章。** 先观察 hooks，再试输入改写、护栏、事件和内存 Trace。

**先预测：** 模型前后、工具前后的日志顺序是什么？修改输入应该使用哪个接口？

**源码入口（固定提交）：**
- [公开生命周期接口](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/lifecycle.py)
- [模型输入过滤](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/run_config.py)
- [Guardrails](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/guardrail.py)
- [流式事件](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/stream_events.py)
- [Tracing](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/tracing/processor_interface.py)

In [17]:
class LifecycleRecorder(RunHooks):
    def __init__(self): self.events = []
    async def on_llm_start(self, context, agent, system_prompt, input_items):
        self.events.append("llm_start")
    async def on_llm_end(self, context, agent, response):
        self.events.append("llm_end")
    async def on_tool_start(self, context, agent, tool):
        self.events.append(f"tool_start:{tool.name}")
    async def on_tool_end(self, context, agent, tool, result):
        self.events.append(f"tool_end:{tool.name}")

hooks = LifecycleRecorder()
hook_agent, hook_model, hook_events = make_weather_agent([
    ModelStep(output=[tool_call("get_weather", {"city": "Shanghai"})]),
    ModelStep(output=[final_message("预设的 Hook 实验答案")]),
])
await Runner.run(hook_agent, "查询天气", hooks=hooks, run_config=OFFLINE)
print(" -> ".join(hooks.events))
assert hooks.events == ["llm_start", "llm_end", "tool_start:get_weather", "tool_end:get_weather", "llm_start", "llm_end"]

llm_start -> llm_end -> tool_start:get_weather -> tool_end:get_weather -> llm_start -> llm_end


这是一次正常非流式工具路径的事件顺序，不代表所有并行、重试或恢复路径。
固定版本的 `on_llm_end` 有延迟处理逻辑，应结合源码阅读。

**动手改：** 在 `on_llm_start` 中额外记录 `agent.name`，观察每次回调作用于谁。

下面用明确的 `call_model_input_filter` 修改模型输入配置，避免把日志回调误当成通用输入转换接口。

In [18]:
from agents.run_config import ModelInputData

def add_study_instruction(call):
    return ModelInputData(
        input=list(call.model_data.input),
        instructions=(call.model_data.instructions or "") + "\nSTUDY_FILTER_MARKER",
    )

filter_model = ScriptedModel([ModelStep(output=[final_message("预设过滤后回答")])])
filter_agent = Agent(name="Filter Demo", instructions="原始指令", model=filter_model)
await Runner.run(filter_agent, "你好", run_config=RunConfig(tracing_disabled=True, call_model_input_filter=add_study_instruction))
print("实际模型边界的指令:", filter_model.calls[0].system_instructions)
assert "STUDY_FILTER_MARKER" in filter_model.calls[0].system_instructions
assert filter_agent.instructions == "原始指令"

实际模型边界的指令: 原始指令
STUDY_FILTER_MARKER


**课后扩展 C：护栏。** 使用顺序输入护栏阻断请求后，模型是否已经被调用？
这里只测试 `run_in_parallel=False`；默认并行检查的时机不同。

In [19]:
from agents import GuardrailFunctionOutput, InputGuardrailTripwireTriggered, input_guardrail

@input_guardrail(run_in_parallel=False)
def only_weather(ctx, agent, user_input):
    return GuardrailFunctionOutput(
        output_info={"topic": "weather"},
        tripwire_triggered="离题" in str(user_input),
    )

guard_model = ScriptedModel([ModelStep(output=[final_message("本次预期不会调用模型")])])
guard_agent = Agent(name="Guard Demo", model=guard_model, input_guardrails=[only_weather])
try:
    await Runner.run(guard_agent, "这是一条离题请求", run_config=OFFLINE)
    raise AssertionError("预期由输入护栏阻断")
except InputGuardrailTripwireTriggered:
    print("输入护栏已阻断")
print("模型调用次数:", len(guard_model.calls))
assert len(guard_model.calls) == 0

输入护栏已阻断
模型调用次数: 0


**课后扩展 D：Streaming。** 流式运行只有文本 delta 吗？
模型按预设事件流输出；我们观察 SDK 对外暴露的事件类型。这里没有网络流式请求。

In [20]:
stream_model = ScriptedModel([ModelStep(output=[final_message("预设的流式文字")])])
stream_agent = Agent(name="Stream Demo", model=stream_model)
stream_result = Runner.run_streamed(stream_agent, "输出一句话", run_config=OFFLINE)
stream_types = []
deltas = []
async for event in stream_result.stream_events():
    stream_types.append(event.type)
    if event.type == "raw_response_event" and event.data.type == "response.output_text.delta":
        deltas.append(event.data.delta)
print("事件类型:", sorted(set(stream_types)))
print("拼接文本:", "".join(deltas))
print("运行结束:", stream_result.is_complete)
print("最终结果:", stream_result.final_output)
assert stream_result.is_complete and stream_result.final_output == "预设的流式文字"
assert "raw_response_event" in stream_types and "run_item_stream_event" in stream_types

事件类型:

 ['agent_updated_stream_event', 'raw_response_event', 'run_item_stream_event']
拼接文本: 预设的流式文字
运行结束: True
最终结果: 预设的流式文字


**课后扩展 E：Trace。** 不上传数据，也能观察 Trace/Span 吗？
下面替换当前内核的 Trace processors，仅把事件写进内存。`finally` 清空处理器，不恢复网络 exporter。
不要在同时承载其他业务的内核中运行本练习；本讲义使用独立学习运行时。

In [21]:
from agents.tracing import TracingProcessor, set_trace_processors

class MemoryTraceProcessor(TracingProcessor):
    def __init__(self): self.events = []
    def on_trace_start(self, trace): self.events.append("trace_start")
    def on_trace_end(self, trace): self.events.append("trace_end")
    def on_span_start(self, span): self.events.append("span_start:" + type(span.span_data).__name__)
    def on_span_end(self, span): self.events.append("span_end:" + type(span.span_data).__name__)
    def force_flush(self): pass
    def shutdown(self): pass

memory_trace = MemoryTraceProcessor()
set_trace_processors([memory_trace])
try:
    trace_model = ScriptedModel([ModelStep(output=[final_message("预设的 Trace 回答")])])
    trace_agent = Agent(name="Trace Demo", model=trace_model)
    await Runner.run(trace_agent, "观察链路", run_config=RunConfig(tracing_disabled=False, trace_include_sensitive_data=False))
    print("只保存在内存的事件:", memory_trace.events)
    assert "trace_start" in memory_trace.events and "trace_end" in memory_trace.events
    assert any(event.startswith("span_start:") for event in memory_trace.events)
finally:
    set_trace_processors([])

只保存在内存的事件: ['trace_start', 'span_start:TaskSpanData', 'span_start:AgentSpanData', 'span_start:TurnSpanData', 'span_end:TurnSpanData', 'span_end:AgentSpanData', 'span_end:TaskSpanData', 'trace_end']


## 第 8 章 · 用边界实验验证 Runtime

**对应笔记：第8章。** 把异常、轮次、交接和恢复的观察汇总。

**先预测：** max_turns=2，但预设三次工具请求：第三次工具会执行吗？

**源码入口（固定提交）：**
- [轮次计数与上限](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/run.py)
- [模拟模型](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/testing/model.py)

In [22]:
MAX_TURNS = 2  # 动手改成 1 或 3；预设工具请求有 3 个
limit_agent, limit_model, limit_events = make_weather_agent([
    ModelStep(output=[tool_call("get_weather", {"city": "Shanghai"})]) for _ in range(3)
])
try:
    await Runner.run(limit_agent, "不断查询天气", max_turns=MAX_TURNS, run_config=OFFLINE)
    raise AssertionError("预期达到轮次上限")
except MaxTurnsExceeded as exc:
    print(type(exc).__name__, str(exc))
print("模型调用次数:", len(limit_model.calls))
print("已执行的工具次数:", len(limit_events))
assert len(limit_model.calls) == MAX_TURNS and len(limit_events) == MAX_TURNS

MaxTurnsExceeded Max turns (2) exceeded
模型调用次数: 2
已执行的工具次数: 2


**观察：** 上限不会撤销已经执行的工具。本单元允许把上限改为 1–3；如果改得更大，模拟脚本会先耗尽，这是另一种错误，不是 SDK 的轮次上限。

**再预测：** 一次响应同时请求两个工具，它们算两个 Turn 吗？
这里观察的是两个工具在一轮完成，不保证并行任务的实际完成顺序。

In [23]:
multi_agent, multi_model, multi_events = make_weather_agent([
    ModelStep(output=[
        tool_call("get_weather", {"city": "Shanghai"}),
        tool_call("get_weather", {"city": "Beijing"}),
    ]),
    ModelStep(output=[final_message("预设的两城天气总结")]),
])
multi_result = await Runner.run(multi_agent, "查询两座城市", run_config=OFFLINE)
print("模型轮次:", len(multi_model.calls))
print("工具执行城市:", multi_events)
print("工具结果数量:", len(tool_outputs(multi_result)))
assert len(multi_model.calls) == 2
assert sorted(multi_events) == ["Beijing", "Shanghai"] and len(tool_outputs(multi_result)) == 2

模型轮次: 2
工具执行城市: ['Shanghai', 'Beijing']
工具结果数量: 2


### 把实测与预期写在一起

| 场景 | 从哪里看证据 | 本讲义的范围 |
|---|---|---|
| 无效参数 / 工具异常 | 工具执行记录、tool output | 当前默认错误处理配置 |
| 直接工具输出 | 模型调用次数、final_output | stop_on_first_tool 的单工具路径 |
| Handoff / as_tool | last_agent、各模型 calls | 预设的正常交接与子任务 |
| 审批恢复 | interruptions、执行计数、模型 calls | 内存中的一次批准与恢复 |
| Hooks / 过滤器 | 回调列表、模型边界的输入 | 正常非流式路径 |
| 护栏 / Streaming / Trace | 模型调用数、事件类型、内存 span | 本节明确给出的配置 |
| 轮次 / 多工具 | calls 与工具执行数 | Turn 与工具数量的区别 |

**仍待单独实验：** 真实模型决策质量、恢复拒绝/再次审批、跨进程持久化、异常取消、Retry、所有工具类型与线上 Trace 导出。
每个预测都要限定到本实验的配置，不把一个正常路径外推为所有路径保证。

## 第 9 章 · 设计取舍：API、SDK 与显式工作流

**对应笔记：第9章。** 用已观察到的职责回答选型问题。

**先预测：** 如果自己只发一次模型请求，需要维护哪些工作？如果要跑多步工具循环，又多了哪些职责？

**源码入口（固定提交）：**
- [模型接口](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/models/interface.py)
- [循环结果](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/run_internal/run_steps.py)

In [24]:
# 先写你的判断再运行。这一节是理解检查，不是框架功能测试。
answers = {
    "模型提出操作意图": "Model",
    "决定下一轮由谁执行": "Runner",
    "保存跨 Run 的聊天历史": "Session",
    "保存审批暂停的执行现场": "RunState",
    "真正执行 Python 业务函数": "Tool callback",
}
expected = dict(zip(answers, ["Model", "Runner", "Session", "RunState", "Tool callback"]))
for responsibility, answer in answers.items():
    print(responsibility, "→", answer)
    assert answer == expected[responsibility], f"再对照笔记：{responsibility}"

模型提出操作意图 → Model
决定下一轮由谁执行 → Runner
保存跨 Run 的聊天历史 → Session
保存审批暂停的执行现场 → RunState
真正执行 Python 业务函数 → Tool callback


**讨论：**

- Responses API 提供模型交互能力；本地函数执行和循环可以由应用自行组织。
- Agents SDK 提供 Agent、Runner 和常用执行机制；本讲义实际运行了其中的限定场景。
- 笔记对 LangGraph 的分析侧重 State/Node/Edge 的显式组织方式。本练习不安装 LangGraph，也不声称完成同版本功能对照。
- 如果业务必须先审批再执行，应该把硬性约束放在哪一层？为什么只写 Prompt 不够？

## 第 10 章 · 从 SDK 观察，回到自己的最小 Runtime

**对应笔记：第10章。** 把完整链路用自己的话讲出来。

**先预测：** 谁负责把 Function Call 变成函数调用？工具结果之后，谁决定继续？

**源码入口（固定提交）：**
- [运行项](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/items.py)
- [执行与决策](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/run_internal/turn_resolution.py)

In [25]:
# 把下面的步骤按你理解的顺序填写，再运行对照。
my_flow = [
    "Agent 配置", "Runner 进入循环", "Model 调用", "解析响应",
    "执行本地工具", "写回工具结果", "下一轮 Model", "最终结果与收尾",
]
for index, step in enumerate(my_flow, 1):
    print(f"{index}. {step}")

print("\n试着不看讲义解释：")
print("1. FunctionTool 为什么同时需要 Schema 和回调？")
print("2. Handoff 为什么可以复用 Function Calling 协议？")
print("3. Session 与 RunState 的生命周期有什么不同？")
print("4. 已有实测能证明什么，还有哪些问题需要新实验？")

1. Agent 配置
2. Runner 进入循环
3. Model 调用
4. 解析响应
5. 执行本地工具
6. 写回工具结果
7. 下一轮 Model
8. 最终结果与收尾

试着不看讲义解释：
1. FunctionTool 为什么同时需要 Schema 和回调？
2. Handoff 为什么可以复用 Function Calling 协议？
3. Session 与 RunState 的生命周期有什么不同？
4. 已有实测能证明什么，还有哪些问题需要新实验？


### 动手写一个最小执行器，再与 SDK 对照

下面真正执行一个精简循环，**不用 Runner**，直接调用模拟模型，再由我们自己的字典查找和函数调用推进任务。
它只支持本例的同步函数与普通文本，省略 Schema 声明、校验、Handoff、审批、护栏、取消和持久化。
模拟响应已预先编排；`tools=[]` 不代表真实模型可以凭空知道你的函数。

**先预测：** 如果去掉 `history.append(function_call_output)`，第二次模型输入会缺少什么？

In [26]:
from agents import ModelSettings
from agents.models.interface import ModelTracing

async def minimal_runtime(model, functions, user_input, max_turns=3):
    history = [{"role": "user", "content": user_input}]
    for turn in range(1, max_turns + 1):
        print(f"自写执行器：第 {turn} 轮")
        response = await model.get_response(
            system_instructions=None, input=history,
            model_settings=ModelSettings(), tools=[], output_schema=None,
            handoffs=[], tracing=ModelTracing.DISABLED,
        previous_response_id=None, conversation_id=None, prompt=None,
        )
        calls = [item for item in response.output if isinstance(item, ResponseFunctionToolCall)]
        history.extend(item.model_dump() for item in response.output)
        if not calls:
            return "".join(part.text for item in response.output if item.type == "message"
                           for part in item.content if part.type == "output_text")
        for call in calls:
            arguments = json.loads(call.arguments)
            value = functions[call.name](**arguments)
            history.append({"type": "function_call_output", "call_id": call.call_id, "output": str(value)})
    raise RuntimeError("最小执行器达到轮次上限")

In [27]:
plain_events = []
def plain_weather(city):
    plain_events.append(city)
    return f"{city}：晴天（固定示例）"

mini_model = ScriptedModel([
    ModelStep(output=[tool_call("get_weather", {"city": "Shanghai"})]),
    ModelStep(output=[final_message("预设的最小执行器回答")]),
])
mini_output = await minimal_runtime(mini_model, {"get_weather": plain_weather}, "查天气")
print("最终结果:", mini_output)
print("实际函数执行记录:", plain_events)
print("第二轮输入是否含工具结果:", any(x.get("type") == "function_call_output" for x in mini_model.calls[1].input))
assert len(mini_model.calls) == 2 and plain_events == ["Shanghai"]
assert any(x.get("type") == "function_call_output" for x in mini_model.calls[1].input)

自写执行器：第 1 轮
自写执行器：第 2 轮
最终结果: 预设的最小执行器回答
实际函数执行记录: ['Shanghai']
第二轮输入是否含工具结果: True


**动手改：** 在自己写的循环中加入一个执行前日志，或修改 `max_turns`，再与 Runner 的结果对照。

这个实验帮助你体会“模型调用”和“运行控制”的分工，不是在几十行代码中复刻完整 SDK。
源码里更长的工具计划、批量调度与状态处理，正是为了处理这里省略的那些能力与边界。

## 可选 · 换成真实模型，观察差异（默认不运行）

这不是主线必做内容。仅当你有可用 OpenAI API 账号、网络和配额，并了解调用可能产生费用时，再手动开启。
密钥通过运行时环境变量 `OPENAI_API_KEY` 提供，不写进 Notebook，也不粘贴到输出。
`REAL_MODEL` 需要使用你的账号实际可用的模型。真实工具仍是固定天气，不查询真实天气 API。

即使模型没有调用工具，也应如实记录；不能沿用 ScriptedModel 的确定性断言。

In [28]:
RUN_REAL_MODEL = False
REAL_MODEL = "gpt-4.1"  # 按账号实际可用模型调整
if RUN_REAL_MODEL:
    import os
    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("请先通过运行时的安全配置提供 OPENAI_API_KEY，不要写入 Notebook。")
    real_events = []
    @function_tool
    def real_demo_weather(city: str) -> str:
        """返回固定教学天气，不是实时查询。"""
        real_events.append(city)
        return f"{city}：晴天，14–20°C（模拟）"
    real_agent = Agent(name="Real Model Demo", instructions="必要时调用天气工具，再回答用户。", model=REAL_MODEL, tools=[real_demo_weather])
    real_result = await Runner.run(real_agent, "上海今天适合户外运动吗？", max_turns=5, run_config=OFFLINE)
    print("工具执行记录:", real_events)
    print("最终回答:", real_result.final_output)
else:
    print("已跳过真实 API 调用；全文主线不需要密钥。")

已跳过真实 API 调用；全文主线不需要密钥。


## 分享后的自查

- [ ] 我能从一个天气请求讲到工具结果回传，而不只列出 SDK 类名。
- [ ] 我修改过至少一个参数，并记录预期与实际结果。
- [ ] 我能解释 Handoff 和 as_tool 的控制权差异。
- [ ] 我能区分 Context、Session 和 RunState。
- [ ] 我知道 hooks、输入过滤、护栏、Streaming 和 Trace 各自的作用范围。
- [ ] 我能说明 ScriptedModel 实验与真实模型行为的区别。

这份练习用于理解 SDK 机制；源码里更复杂的异常、取消、并发与持久化边界，仍要继续逐条研究。

**提交或分享副本之前：清空所有输出，确认没有密钥、个人数据或内部业务信息。**
本讲义的可分享版本不保存旧报错，现场输出由听众自己运行产生。